# HPO Analysis

Визуализация результатов подбора гиперпараметров (Optuna).

Данные: `output/hpo_optuna.db` (SQLite), `output/hpo_architecture.json`, `output/hpo_training.json`

In [ ]:
import json
from pathlib import Path

import optuna
import plotly.io as pio
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate,
    plot_slice,
)

pio.renderers.default = "notebook"

OUTPUT_DIR = Path("../output")
STORAGE = f"sqlite:///{OUTPUT_DIR / 'hpo_optuna.db'}"

## 1. Этап 1: Архитектура

In [ ]:
arch_study = optuna.load_study(study_name="er_architecture", storage=STORAGE)
print(f"Trials: {len(arch_study.trials)}")
print(f"Best val_loss: {arch_study.best_value:.4f}")
print(f"Best params:")
for k, v in arch_study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
plot_optimization_history(arch_study, target_name="val_loss")

In [ ]:
plot_param_importances(arch_study)

In [ ]:
plot_parallel_coordinate(arch_study)

In [ ]:
plot_slice(arch_study)

### Таблица всех trial-ов (архитектура)

In [ ]:
df_arch = arch_study.trials_dataframe()
df_arch = df_arch.sort_values("value").reset_index(drop=True)
cols = [c for c in df_arch.columns if c.startswith("params_") or c in ("number", "value", "state", "duration")]
df_arch[cols].head(20)

## 2. Этап 2: Обучение

In [ ]:
try:
    train_study = optuna.load_study(study_name="er_training", storage=STORAGE)
    print(f"Trials: {len(train_study.trials)}")
    print(f"Best val_loss: {train_study.best_value:.4f}")
    print(f"Best params:")
    for k, v in train_study.best_params.items():
        print(f"  {k}: {v}")
except KeyError:
    print("Этап 2 ещё не запускался")
    train_study = None

In [ ]:
if train_study:
    plot_optimization_history(train_study, target_name="val_loss").show()

In [ ]:
if train_study:
    plot_param_importances(train_study).show()

In [ ]:
if train_study:
    plot_slice(train_study).show()

In [ ]:
if train_study:
    df_train = train_study.trials_dataframe()
    df_train = df_train.sort_values("value").reset_index(drop=True)
    cols = [c for c in df_train.columns if c.startswith("params_") or c in ("number", "value", "state", "duration")]
    display(df_train[cols].head(20))

## 3. Итоговая конфигурация

In [ ]:
arch_path = OUTPUT_DIR / "hpo_architecture.json"
train_path = OUTPUT_DIR / "hpo_training.json"

if arch_path.exists():
    arch = json.load(open(arch_path))
    print("=== Лучшая архитектура ===")
    print(f"val_loss: {arch['best_value']:.4f}")
    for k, v in arch["best_params"].items():
        print(f"  {k}: {v}")

if train_path.exists():
    train = json.load(open(train_path))
    print("\n=== Лучшие параметры обучения ===")
    print(f"val_loss: {train['best_value']:.4f}")
    for k, v in train["best_training"].items():
        print(f"  {k}: {v}")
    
    print("\n=== Финальная конфигурация для EntityResolutionConfig ===")
    final = {**train["best_arch"], **train["best_training"]}
    for k, v in final.items():
        print(f"  {k}: {v}")